# DiffSinger Miadu Fine-tuning (100步高頻保存版)
**修正內容**：
- **高頻保存**：每 100 步保存一次權重，防止 Colab 斷線丟進度。
- **斷點續練**：重啟後自動找回最新進度。
- **同步修復**：強制將存檔路徑指向 Google Drive。

## 第一步：掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 第二步：獲取/更新代碼與環境配置

In [ ]:
%cd /content/
import os
if not os.path.exists('diffsinger-repo'):
    !git clone https://github.com/shihte/DiffSinger-Miadu-Colab.git diffsinger-repo
else:
    print('| 偵測到已有代碼，正在執行更新...')
    %cd diffsinger-repo
    !git pull
    %cd ..

%cd diffsinger-repo
!pip install --upgrade setuptools pip wheel
!pip install praat-parselmouth pyloudnorm pypinyin pycwt pyworld lightning-flash==0.7.0

## 第三步：數據同步與【強制路徑對齊】

In [ ]:
import os
import torch

drive_root = "/content/drive/MyDrive/DiffSinger_Colab"
if not os.path.exists(drive_root):
    drive_root = "/content/drive/MyDrive/DiffSinger _Colab"

print(f'| 使用 Drive 目錄: {drive_root}')

# --- 【關鍵修正】強制替換 checkpoints 資料夾為軟連結 ---
target_link = "/content/diffsinger-repo/checkpoints"
drive_target = f"{drive_root}/checkpoints_finetune"

!mkdir -p "{drive_target}"

if os.path.exists(target_link) and not os.path.islink(target_link):
    print('| 正在遷移本地 Checkpoints 到 Drive...')
    !cp -rv "{target_link}"/* "{drive_target}/" 2>/dev/null || true
    !rm -rf "{target_link}"

if not os.path.exists(target_link):
    !ln -s "{drive_target}" "{target_link}"
    print('| 已成功建立 Drive 同步連結 ✅')
else:
    print('| 同步連結已存在 ✅')

# ---------------------------------------------------

def robust_sync(src_dir, dest_path, min_size_mb=1):
    dest_dir = os.path.dirname(dest_path)
    !mkdir -p "{dest_dir}"
    if not os.path.exists(dest_path) or os.path.getsize(dest_path) < min_size_mb * 1024 * 1024:
        print(f"| 正在同步數據: {os.path.basename(src_dir)}")
        !cp -rv "{src_dir}" "{dest_dir}/.."
    else:
        print(f"| 已跳過完整數據: {os.path.basename(dest_path)}")

robust_sync(f"{drive_root}/1117_opencpop_ds1000_strict_pinyin", 
            "checkpoints/1117_opencpop_ds1000_strict_pinyin/model_ckpt_steps_200000.ckpt", 800)
robust_sync(f"{drive_root}/nsf_hifigan_44.1k_hop512_128bin_2024.02", 
            "checkpoints/nsf_hifigan_44.1k_hop512_128bin_2024.02/model_ckpt_steps_160000.ckpt", 10)
robust_sync(f"{drive_root}/miadu_finetune", 
            "data/binary/miadu_finetune/train.data", 1000)

print('| 數據準備完成！所有存檔將直接寫入 Google Drive。')

## 第四步：啟動訓練

In [ ]:
import os
os.environ['PYTHONPATH'] = "."
!python tasks/run.py --config usr/configs/midi/e2e/miadu_finetune.yaml --exp_name miadu_finetune_v1